# 03 – Preprocessing: Filtrado de Variables

**Proyecto:** Encuesta Permanente de Empleo Nacional (EPEN)  
**Objetivo:** Eliminar variables que no aportan valor predictivo: variables con alta cardinalidad, quasi-constantes, identificadores o con excesivos valores faltantes.

In [1]:
import pandas as pd
import numpy as np
import os

# ─── Cargar datos ─────────────────────────────────────────────────────────────
PROC_PATH = os.path.join('..', 'data', 'processed', 'epen_missing_handled.csv')
try:
    df = pd.read_csv(PROC_PATH)
except FileNotFoundError:
    np.random.seed(42)
    n = 1000
    df = pd.DataFrame({
        'id_persona': range(n),                                    # Identificador (debe eliminarse)
        'edad': np.random.randint(14, 70, n),
        'sexo': np.random.choice(['Hombre', 'Mujer'], n),
        'nivel_educativo': np.random.choice(
            ['Sin instrucción', 'Primaria', 'Secundaria', 'Preparatoria', 'Universidad', 'Posgrado'], n
        ),
        'estado_civil': np.random.choice(['Soltero', 'Casado', 'Unión libre', 'Divorciado', 'Viudo'], n),
        'ingreso_mensual': np.random.exponential(8000, n).round(2),
        'horas_trabajadas': np.random.randint(0, 60, n),
        'tipo_empleo': np.random.choice(['Formal', 'Informal', 'Sin empleo'], n),
        'sector': np.random.choice(['Agricultura', 'Industria', 'Comercio', 'Servicios', 'Gobierno'], n),
        'constante': 1,                                           # Quasi-constante (debe eliminarse)
        'condicion_actividad': np.random.choice(
            ['Ocupado', 'Desocupado', 'No PEA'], n, p=[0.60, 0.10, 0.30]
        ),
        'target_desocupado': np.random.choice([0, 1], n, p=[0.90, 0.10]),
    })

print(f'Dataset cargado: {df.shape[0]:,} filas × {df.shape[1]} columnas')
print('Columnas:', df.columns.tolist())

Dataset cargado: 24,054 filas × 138 columnas
Columnas: ['ANIO', 'MES', 'CONGLOMERADO', 'MUESTRA', 'SELVIV', 'HOGAR', 'REGION', 'LLAVE_PANEL', 'ESTRATO', 'C201', 'C203', 'C204', 'C205', 'C206', 'C207', 'C208', 'C300n', 'NROINF', 'C301_DIA', 'C301_MES', 'C301_ANIO', 'C303', 'C304', 'C305', 'C306_1', 'C306_2', 'C306_3', 'C306_4', 'C306_5', 'C306_6', 'C306_7', 'C306_8', 'C306_9', 'C306_10', 'C306_10A', 'C306_11', 'C306A', 'C308_COD', 'C309_COD', 'C310', 'C311', 'C312', 'C313', 'C317', 'C317A', 'C318_1', 'C318_2', 'C318_3', 'C318_4', 'C318_5', 'C318_6', 'C318_7', 'C318_T', 'C328_T', 'whoraT', 'C330', 'C331', 'C333', 'C334', 'P209H', 'C335', 'C338', 'C339_1', 'C341_T', 'C342', 'C344', 'C345_1', 'C347_T', 'C348', 'C350', 'C352', 'C353', 'C354', 'C355', 'C356', 'C357_I', 'C358', 'C359', 'SEGURO1', 'C361_1', 'C362_1', 'C361_2', 'C362_2', 'C361_3', 'C362_3', 'C361_4', 'C362_4', 'C361_5', 'C362_5', 'C361_6', 'C362_6', 'C361_7', 'C362_7', 'C361_8', 'C362_8', 'C364_1', 'C365_1', 'C364_2', 'C365_2',

C:\Users\guill\AppData\Local\Temp\ipykernel_14144\1333699246.py:8: DtypeWarning: Columns (8) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(PROC_PATH)


## 1. Variables a eliminar manualmente

Estas variables se eliminan por criterio de dominio (no tienen poder predictivo):

In [2]:
# Variables identificadoras o redundantes
drop_manual = [col for col in ['id_persona', 'condicion_actividad'] if col in df.columns]
print('Eliminando manualmente:', drop_manual)
df_filtered = df.drop(columns=drop_manual)

Eliminando manualmente: []


## 2. Variables quasi-constantes

In [3]:
THRESHOLD_VARIANCE = 0.01
quasi_constant = []
for col in df_filtered.select_dtypes(include=np.number).columns:
    if col == 'target_desocupado':
        continue
    # Proporción del valor más frecuente
    top_freq = df_filtered[col].value_counts(normalize=True).iloc[0]
    if top_freq > (1 - THRESHOLD_VARIANCE):
        quasi_constant.append(col)
        print(f'  Quasi-constante: {col} (freq. top = {top_freq:.3f})')

df_filtered = df_filtered.drop(columns=quasi_constant)
print(f'\nVariables eliminadas por quasi-constancia: {quasi_constant}')

  Quasi-constante: ANIO (freq. top = 1.000)
  Quasi-constante: MUESTRA (freq. top = 0.999)
  Quasi-constante: REGION (freq. top = 1.000)
  Quasi-constante: C204 (freq. top = 0.996)
  Quasi-constante: C361_6 (freq. top = 0.999)
  Quasi-constante: C361_7 (freq. top = 1.000)
  Quasi-constante: C361_8 (freq. top = 1.000)
  Quasi-constante: C364_3 (freq. top = 0.999)
  Quasi-constante: C375_1 (freq. top = 0.997)
  Quasi-constante: C375_2 (freq. top = 0.999)
  Quasi-constante: C375_3 (freq. top = 1.000)
  Quasi-constante: C375_4 (freq. top = 0.999)
  Quasi-constante: C375_5 (freq. top = 0.998)
  Quasi-constante: C375_6 (freq. top = 0.998)
  Quasi-constante: OCUP300 (freq. top = 1.000)
  Quasi-constante: RESIDENT (freq. top = 1.000)
  Quasi-constante: OCUP300_num (freq. top = 1.000)
  Quasi-constante: whoraT_missing_flag (freq. top = 1.000)
  Quasi-constante: C318_T_missing_flag (freq. top = 1.000)

Variables eliminadas por quasi-constancia: ['ANIO', 'MUESTRA', 'REGION', 'C204', 'C361_6', 'C3

## 3. Variables con excesivos valores faltantes (>50%)

In [4]:
THRESHOLD_MISSING = 0.50
high_missing = df_filtered.columns[df_filtered.isnull().mean() > THRESHOLD_MISSING].tolist()
if high_missing:
    print(f'Variables con >{THRESHOLD_MISSING*100:.0f}% de nulos: {high_missing}')
    df_filtered = df_filtered.drop(columns=high_missing)
else:
    print('No se encontraron variables con excesivos valores faltantes.')

Variables con >50% de nulos: ['fa_ond24', 'fa_efm24', 'fa_amj24', 'fa_jas24']


In [5]:
print(f'\nColumnas finales ({df_filtered.shape[1]}): {df_filtered.columns.tolist()}')

os.makedirs(os.path.join('..', 'data', 'processed'), exist_ok=True)
df_filtered.to_csv(os.path.join('..', 'data', 'processed', 'epen_filtered.csv'), index=False)
print('Dataset guardado: epen_filtered.csv')


Columnas finales (115): ['MES', 'CONGLOMERADO', 'SELVIV', 'HOGAR', 'LLAVE_PANEL', 'ESTRATO', 'C201', 'C203', 'C205', 'C206', 'C207', 'C208', 'C300n', 'NROINF', 'C301_DIA', 'C301_MES', 'C301_ANIO', 'C303', 'C304', 'C305', 'C306_1', 'C306_2', 'C306_3', 'C306_4', 'C306_5', 'C306_6', 'C306_7', 'C306_8', 'C306_9', 'C306_10', 'C306_10A', 'C306_11', 'C306A', 'C308_COD', 'C309_COD', 'C310', 'C311', 'C312', 'C313', 'C317', 'C317A', 'C318_1', 'C318_2', 'C318_3', 'C318_4', 'C318_5', 'C318_6', 'C318_7', 'C318_T', 'C328_T', 'whoraT', 'C330', 'C331', 'C333', 'C334', 'P209H', 'C335', 'C338', 'C339_1', 'C341_T', 'C342', 'C344', 'C345_1', 'C347_T', 'C348', 'C350', 'C352', 'C353', 'C354', 'C355', 'C356', 'C357_I', 'C358', 'C359', 'SEGURO1', 'C361_1', 'C362_1', 'C361_2', 'C362_2', 'C361_3', 'C362_3', 'C361_4', 'C362_4', 'C361_5', 'C362_5', 'C362_6', 'C362_7', 'C362_8', 'C364_1', 'C365_1', 'C364_2', 'C365_2', 'C365_3', 'C364_4', 'C365_4', 'C366', 'C366_1', 'C366_2', 'C376', 'C377', 'I339_1', 'D341_T', 'I